In [ ]:
MIN_SCORE = 1000
TOP_N = 40
PER_TEAM = 0
PREVIEW_N = 10
MAX_NEW_DAYS = 2

OUT_DIR = "/kaggle/working/top_replays"
PRIOR_DS = "/kaggle/input/datasets/seshurajup/kaggriculture-top-replays-download-dataset-daily"
INDEX_DS = "kaggle/kaggriculture-episodes-index"

## What is already done

The day folders we already have are the state, so each run only fetches dates that are missing.

In [ ]:
import logging, warnings, io, contextlib
warnings.filterwarnings("ignore")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import kaggle_environments
for name in list(logging.root.manager.loggerDict):
    if name.startswith("kaggle_environments"):
        logging.getLogger(name).setLevel(logging.ERROR)
logging.getLogger("kaggle_environments").setLevel(logging.ERROR)

import os, io, glob, json, shutil, zipfile, subprocess, collections
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pyarrow as pa
import pyarrow.parquet as pq

plt.rcParams.update({"figure.dpi": 90, "font.size": 10, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25})

def sh(cmd, timeout=3600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)

os.makedirs(OUT_DIR, exist_ok=True)

done = set()
if os.path.isdir(PRIOR_DS):
    for d in sorted(os.listdir(PRIOR_DS)):
        src = os.path.join(PRIOR_DS, d)
        if os.path.isdir(src) and d.startswith("20"):
            dst = os.path.join(OUT_DIR, d)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
            done.add(d)

IDX = "/kaggle/working/_index"
os.makedirs(IDX, exist_ok=True)
sh(f"kaggle datasets download -d {INDEX_DS} -p {IDX} --unzip --force")
manifest = pd.read_csv(os.path.join(IDX, "manifest.csv")).sort_values("date", ascending=False)

todo = [r for _, r in manifest.iterrows() if r["date"] not in done][:MAX_NEW_DAYS]
print(f"have: {sorted(done) or 'nothing yet'}")
print(f"new : {[r['date'] for r in todo] or 'up to date'}")
manifest.head(6)

## Fetch and filter

Selection is on the weaker player's score, so both agents in a kept game are strong. `PER_TEAM` caps how
many wins one author can contribute: leave it 0 for the raw top N, or set it to 1 for one game per author.

In [ ]:
SCHEMA = pa.schema([("episode_id", pa.int64()), ("avg_score", pa.float64()),
                    ("min_score", pa.float64()), ("date", pa.string()),
                    ("replay_json", pa.string())])

def write_day(date, sel, read_json):
    sel, winners = cap_per_team(sel, read_json)
    day_dir = os.path.join(OUT_DIR, date)
    os.makedirs(day_dir, exist_ok=True)
    out = os.path.join(day_dir, "replays.parquet")
    writer = pq.ParquetWriter(out, SCHEMA, compression="zstd", compression_level=10)
    raw = 0
    recs = list(sel.iterrows())
    try:
        for start in range(0, len(recs), 10):
            ids, avg, mn, dts, blobs = [], [], [], [], []
            for _, r in recs[start:start + 10]:
                txt = read_json(r["episode_id"])
                if txt is None:
                    continue
                raw += len(txt)
                ids.append(int(r["episode_id"]))
                avg.append(float(r["avg_score"]))
                mn.append(float(r["min_score"]))
                dts.append(date)
                blobs.append(txt)
            if ids:
                writer.write_table(pa.table({"episode_id": ids, "avg_score": avg, "min_score": mn,
                                             "date": dts, "replay_json": blobs}, schema=SCHEMA))
    finally:
        writer.close()
    out_csv = sel.assign(date=date)
    if winners:
        out_csv = out_csv.assign(winner=out_csv["episode_id"].map(winners))
    out_csv.to_csv(os.path.join(day_dir, "scores.csv"), index=False)
    size = os.path.getsize(out)
    print(f"  {date}: {len(sel)} replays, {raw/1e6:.0f} MB -> {size/1e6:.1f} MB ({raw/max(size,1):.0f}x)")

def pick(m):
    m = m.copy()
    m["min_score"] = pd.to_numeric(m["min_score"], errors="coerce")
    m["avg_score"] = pd.to_numeric(m["avg_score"], errors="coerce")
    return m[m["min_score"] >= MIN_SCORE].sort_values("min_score", ascending=False).head(
        TOP_N if not PER_TEAM else TOP_N * 6)

def cap_per_team(sel, read_json):
    if not PER_TEAM:
        return sel, None
    kept, seen, names = [], collections.Counter(), {}
    for _, r in sel.iterrows():
        txt = read_json(r["episode_id"])
        if txt is None:
            continue
        info = (json.loads(txt).get("info") or {})
        team = (info.get("TeamNames") or ["?"])[0]
        rw = json.loads(txt).get("rewards") or [0, 0]
        tn = info.get("TeamNames") or ["?", "?"]
        winner = tn[0 if rw[0] >= rw[1] else 1] if len(tn) > 1 else tn[0]
        if seen[winner] >= PER_TEAM:
            continue
        seen[winner] += 1
        names[r["episode_id"]] = winner
        kept.append(r)
        if len(kept) >= TOP_N:
            break
    return pd.DataFrame(kept), names

def process(row):
    date, slug = row["date"], row["daily_dataset_slug"]
    mounted = f"/kaggle/input/{slug}"
    if os.path.isdir(mounted) and os.path.exists(os.path.join(mounted, "manifest.csv")):
        sel = pick(pd.read_csv(os.path.join(mounted, "manifest.csv")))
        if len(sel):
            write_day(date, sel, lambda e: open(os.path.join(mounted, f"{e}.json")).read()
                      if os.path.exists(os.path.join(mounted, f"{e}.json")) else None)
        return sel

    zp = None
    if os.path.isdir(mounted):
        z = glob.glob(os.path.join(mounted, "*.zip"))
        zp = z[0] if z else None
    if zp is None:
        tmp = f"/kaggle/working/_day_{date}"
        os.makedirs(tmp, exist_ok=True)
        print(f"  downloading {slug}")
        sh(f"kaggle datasets download -d kaggle/{slug} -p {tmp}")
        z = glob.glob(os.path.join(tmp, "*.zip"))
        zp = z[0] if z else None
    if zp is None:
        print(f"  could not get {slug}")
        return None

    zf = zipfile.ZipFile(zp)
    sel = pick(pd.read_csv(io.BytesIO(zf.read("manifest.csv"))))
    if len(sel):
        write_day(date, sel, lambda e: zf.read(f"{e}.json").decode())
    shutil.rmtree(f"/kaggle/working/_day_{date}", ignore_errors=True)
    return sel

for r in todo:
    process(r)

## The corpus

Only the score columns are read here; a replay is loaded on demand.

In [ ]:
parts = []
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*", "replays.parquet"))):
    m = pq.read_table(f, columns=["episode_id", "avg_score", "min_score", "date"]).to_pandas()
    m["file"] = f
    parts.append(m)

corpus = pd.concat(parts, ignore_index=True).sort_values("min_score", ascending=False)
size = sum(os.path.getsize(p) for p in corpus["file"].unique())
print(f"{len(corpus)} replays, {corpus['date'].nunique()} days, {size/1e6:.1f} MB")

def load_replay(episode_id, file):
    t = pq.read_table(file, filters=[("episode_id", "==", int(episode_id))], columns=["replay_json"])
    return json.loads(t.column("replay_json")[0].as_py())

view = corpus.head(PREVIEW_N)[["date", "episode_id", "avg_score", "min_score"]]
view.style.background_gradient(subset=["avg_score", "min_score"], cmap="Greens").format(
    {"avg_score": "{:.0f}", "min_score": "{:.0f}"})

## Load the top games

Validation episodes run one submission against itself and end with identical banks, so they are dropped.

In [ ]:
clean = []
for _, r in corpus.head(PREVIEW_N).iterrows():
    ep = load_replay(r["episode_id"], r["file"])
    rw = ep.get("rewards") or [0, 0]
    if rw[0] == rw[1]:
        continue
    names = (ep.get("info") or {}).get("TeamNames") or []
    who = names[0 if rw[0] >= rw[1] else 1] if len(names) > 1 else "?"
    clean.append((f"{int(r['avg_score'])}_{who}", ep))

def day_series(ep, seat, key="money"):
    out, seen = [], set()
    for st in ep["steps"]:
        o = (st[0] or {}).get("observation") or {}
        fl = o.get("farms") or []
        if len(fl) <= seat:
            continue
        d = int(o.get("day", 0) or 0)
        if d in seen:
            continue
        seen.add(d)
        out.append((d, fl[seat][key]))
    return list(zip(*out)) if out else ([], [])

def prices_series(ep, product):
    out, seen = [], set()
    for st in ep["steps"]:
        o = (st[0] or {}).get("observation") or {}
        d = int(o.get("day", 0) or 0)
        if d in seen:
            continue
        p = ((o.get("market") or {}).get("prices") or {}).get(product)
        if p is None:
            continue
        seen.add(d)
        out.append((d, p))
    return list(zip(*out)) if out else ([], [])

print(f"{len(clean)} games ready")

## Both banks per game

The gap between the two lines shows where the game was actually decided.

In [ ]:
n = min(6, len(clean))
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True)
for ax, (label, ep) in zip(axes.ravel(), clean[:n]):
    rw = ep.get("rewards") or [0, 0]
    win = 0 if rw[0] >= rw[1] else 1
    d0, m0 = day_series(ep, 0)
    d1, m1 = day_series(ep, 1)
    ax.plot(d0, m0, lw=2 if win == 0 else 1.2, color="#2b8a3e" if win == 0 else "#adb5bd",
            label=f"seat0 {rw[0]:,.0f}")
    ax.plot(d1, m1, lw=2 if win == 1 else 1.2, color="#2b8a3e" if win == 1 else "#adb5bd",
            label=f"seat1 {rw[1]:,.0f}")
    if d0 and d1:
        k = min(len(m0), len(m1))
        ax.fill_between(list(d0)[:k], list(m0)[:k], list(m1)[:k], alpha=0.12, color="#2b8a3e")
    ax.set_title(label.split("_")[0])
    ax.legend(fontsize=7, frameon=False)
for ax in axes.ravel()[n:]:
    ax.axis("off")
fig.supxlabel("day")
fig.supylabel("bank")
plt.tight_layout()
plt.show()

## The shared market

Both farms sell into one price curve, so the premium goods collapse as volume arrives.

In [ ]:
label, ep = clean[0]
fig, ax = plt.subplots(figsize=(11, 4.5))
for prod, col in [("MILK", "#1c7ed6"), ("STRAWBERRY", "#e03131"), ("MELON", "#f08c00"),
                  ("WOOL", "#7048e8"), ("WHEAT", "#868e96")]:
    d, p = prices_series(ep, prod)
    if d:
        ax.plot(d, p, label=prod, color=col, lw=1.8)
ax.set_xlabel("day")
ax.set_ylabel("unit price")
ax.set_title(f"shared market — {label}")
ax.legend(frameon=False, ncol=5)
plt.tight_layout()
plt.show()

## Watch the game

Both farms side by side as it played: the big dot is the farmer, small dots are hired hands, green dots are
banked yield and orange circles are animals. Dark quadrants are land the agent never bought, so they stay
locked for the whole season.

In [ ]:
PLAY_RANK = 0
EVERY = 6

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

label, ep = clean[PLAY_RANK]
rw = ep.get("rewards") or [0, 0]
win = 0 if rw[0] >= rw[1] else 1
steps = ep["steps"]
print(f"{label}   seat0 {rw[0]:,.0f}   seat1 {rw[1]:,.0f}   -> seat{win} wins")

TILE = {"EMPTY": "#d9c8a9", "LOCKED": "#495057", "PLANT": "#51cf66", "WEED": "#9c7a1a",
        "COOP": "#ffd43b", "PASTURE": "#ff922b"}
TRACKED = [("MILK", "#1c7ed6"), ("STRAWBERRY", "#e03131"), ("MELON", "#f08c00"), ("WOOL", "#7048e8")]

fig = plt.figure(figsize=(10, 4.8), dpi=68)
grid = fig.add_gridspec(2, 2, height_ratios=[6, 1.5], hspace=0.3, wspace=0.1)
boards = [fig.add_subplot(grid[0, 0]), fig.add_subplot(grid[0, 1])]
market = fig.add_subplot(grid[1, :])
hist = {p: [] for p, _ in TRACKED}
days = []

def frame(i):
    obs = (steps[i][0] or {}).get("observation") or {}
    farms = obs.get("farms") or []
    for seat in (0, 1):
        ax = boards[seat]
        ax.clear()
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        if seat >= len(farms):
            continue
        f = farms[seat]
        size = len(f["tiles"])
        for y, row in enumerate(f["tiles"]):
            for x, t in enumerate(row):
                if isinstance(t, dict):
                    kind = t.get("kind", "EMPTY")
                    ax.add_patch(mpatches.Rectangle((x, -y), 1, 1, facecolor=TILE.get(kind, "#f8f9fa"),
                                                    edgecolor="white", lw=0.35))
                    if t.get("animal"):
                        ax.add_patch(mpatches.Circle((x + 0.5, -y + 0.5), 0.28, facecolor="#d9480f",
                                                     ec="white", lw=0.5))
                    elif kind == "PLANT" and t.get("yield_units", 0):
                        r = 0.12 + 0.04 * min(t["yield_units"], 4)
                        ax.add_patch(mpatches.Circle((x + 0.5, -y + 0.5), r, facecolor="#2b8a3e", ec="none"))
                else:
                    kind = "LOCKED" if t == "LOCKED" else "EMPTY"
                    ax.add_patch(mpatches.Rectangle((x, -y), 1, 1, facecolor=TILE[kind],
                                                    edgecolor="white", lw=0.35))
        fx, fy = f["farmer"]
        ax.add_patch(mpatches.Circle((fx + 0.5, -fy + 0.5), 0.34, facecolor="#1864ab", ec="white",
                                     lw=1.0, zorder=5))
        for h in (f.get("hands") or []):
            ax.add_patch(mpatches.Circle((h[0] + 0.5, -h[1] + 0.5), 0.20, facecolor="#4dabf7",
                                         ec="white", lw=0.7, zorder=4))
        half = size // 2
        for qx, qy, qn in ((0, 0, "NW"), (half, 0, "NE"), (0, half, "SW"), (half, half, "SE")):
            if f["tiles"][qy][qx] == "LOCKED":
                ax.text(qx + half / 2, -(qy + half / 2), f"{qn}\nlocked", ha="center", va="center",
                        fontsize=7, color="#ced4da", zorder=7)
        ax.axvline(half, color="#adb5bd", lw=0.9, zorder=6)
        ax.axhline(-half + 1, color="#adb5bd", lw=0.9, zorder=6)
        ax.set_xlim(0, size)
        ax.set_ylim(-size, 1)
        ax.set_aspect("equal")
        nq = len(f.get("unlocked_quadrants") or [])
        ax.set_title(f"seat{seat}   {f['money']:,.0f}   crew {len(f.get('hands') or [])}   {nq}/4 land",
                     fontsize=9, color="#2b8a3e" if seat == win else "#495057")

    day = int(obs.get("day", 0) or 0)
    prices = ((obs.get("market") or {}).get("prices") or {})
    days.append(day)
    for p, _ in TRACKED:
        hist[p].append(prices.get(p, 0))
    market.clear()
    for p, col in TRACKED:
        market.plot(days, hist[p], color=col, lw=1.3, label=p)
    market.set_xlim(0, 30)
    market.tick_params(labelsize=7)
    market.legend(fontsize=7, frameon=False, ncol=4, loc="upper right")
    market.spines[["top", "right"]].set_visible(False)
    fig.suptitle(f"day {day} of 30   ·   hour {obs.get('hour', 0)}   ·   frame {frames_done[0]}/{n_frames}", fontsize=11, y=0.98)
    frames_done[0] += 1

frame_ids = list(range(0, len(steps), EVERY))
n_frames = len(frame_ids)
frames_done = [1]
anim = FuncAnimation(fig, frame, frames=frame_ids, interval=70)
plt.close(fig)
HTML(anim.to_jshtml())

## Play it

The interactive player is about 40 MB for one game, which is too big to embed, so it is written to the
output where you can open or download it.

In [ ]:
env = kaggle_environments.make("kaggriculture", configuration=ep.get("configuration") or {},
                               steps=ep["steps"], debug=False)
html = env.render(mode="html", width=980, height=700)
path = f"/kaggle/working/replay_{label}.html"
with open(path, "w") as f:
    f.write(html)
print(f"{path}  ({len(html)/1e6:.0f} MB)")

## Strategy fingerprint

What each winner spent its 719 decisions on.

In [ ]:
MOVES = {"NORTH", "SOUTH", "EAST", "WEST"}
rows = []
for label_, ep_ in clean:
    rw = ep_.get("rewards") or [0, 0]
    w = 0 if rw[0] >= rw[1] else 1
    acts = collections.Counter()
    crew = 0
    first_land = None
    for t, st in enumerate(ep_["steps"]):
        o = (st[0] or {}).get("observation") or {}
        fl = o.get("farms") or []
        if len(fl) > w:
            crew = max(crew, len(fl[w].get("hands") or []))
        a = (st[w] or {}).get("action") or {}
        for od in (a.get("market") or []):
            if isinstance(od, list) and od and od[0] == "BUY_LAND" and first_land is None:
                first_land = t // 24
        for u in [a.get("farmer")] + list(a.get("hands") or []):
            if u:
                acts["MOVE" if u[0] in MOVES else u[0]] += 1
    obs = (ep_["steps"][-1][w] or {}).get("observation") or {}
    fl = obs.get("farms") or []
    f = fl[w] if len(fl) > w else {"tiles": []}
    tl = [x for row in f.get("tiles", []) for x in row]
    rows.append({"score": label_.split("_")[0], "author": label_.split("_", 1)[1], "bank": int(rw[w] or 0), "land day": first_land,
                 "crew": crew, "quads": len(f.get("unlocked_quadrants") or []),
                 "animals": sum(1 for x in tl if isinstance(x, dict) and x.get("animal")),
                 "WATER": acts.get("WATER", 0), "FEED": acts.get("FEED", 0),
                 "CARE": acts.get("CARE", 0), "PASS": acts.get("PASS", 0)})

df = pd.DataFrame(rows).sort_values("bank", ascending=False)
df.style.background_gradient(subset=["bank", "animals", "CARE"], cmap="Greens").format({"bank": "{:,.0f}"})

## Saving

Save & Run All, then attach `/kaggle/working/top_replays` as a dataset and mount it back next run so the
corpus keeps growing.

In [ ]:
for d in sorted(glob.glob(os.path.join(OUT_DIR, "*"))):
    f = os.path.join(d, "replays.parquet")
    if os.path.exists(f):
        print(f"{os.path.basename(d)}  {pq.read_metadata(f).num_rows} replays  "
              f"{os.path.getsize(f)/1e6:.1f} MB")